In [ ]:
# Dataset completo: https://www.kaggle.com/c/tensorflow-speech-recognition-challenge/data
# Possui mais de 3 Gb, portanto disponibilizarei apenas as 3 pastas (bird, cat, dog)

In [ ]:
import os
import librosa   # pip install librosa 
import IPython.display as ipd # biblioteca para rodar áudios
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Primeiro, vamos carregar uma amostra de áudio para dentro do jupyter notebook
diretorio = 'C:/Users/Natanael/Documents/DidaticaTech/train/audio/bird/'
amostra, sample_rate = librosa.load(diretorio+'00b01445_nohash_0.wav', sr = 16000) # carrega o áudio especificado com 16.000 frames por segundo. Em 'amostra' terei a magnitude de cada ponto no espaço, e em sample_rate terei o sr informado.
fig = plt.figure(figsize=(14, 8))
ax1 = fig.add_subplot(2,1,1)
ax1.set_title('Amostra do áudio ' + '00b01445_nohash_0.wav')
ax1.set_xlabel('Tempo')
ax1.set_ylabel('Amplitude')
ax1.plot(np.linspace(0, 1, sample_rate), amostra) 

# np.linspace gera números igualmente espaçados dentro de um intervalo. Por exemplo: np.linspace(2.0, 3.0, 5) irá gerar 5 números
# igualmente espaçados localizados entre 2.0 e 3.0, que seriam: [2., 2.25, 2.5, 2.75, 3.].
# Nesse caso, estou criando no eixo x sample_rate pontos igualmente espaçados emtre zero e 1, plotando os valores das magnitures
# da amostra em questão.

In [ ]:
len(amostra)

In [ ]:
# Criando e visualizando o espectro MFCC:
mfcc = librosa.feature.mfcc(y=amostra, hop_length=512, n_mfcc=20) # hop_length é quantas vezes irei subdividir a amostra para criar as features do MFCC
# O cálculo é (seconds x frames/seconds)/hop_length. Por exemplo, se hop_length=512 (default), tenho que 1 x 16000/512 = 32

In [ ]:
mfcc.shape

In [ ]:
# Mostrando o MFCC:
import librosa.display
plt.figure(figsize=(8, 5))
librosa.display.specshow(mfcc, x_axis='time', sr=16000)
plt.colorbar()
plt.title('MFCC')
plt.tight_layout()
plt.show()

In [ ]:
# Ouvindo uma amostra dentro do jupyter notebook:
ipd.Audio(amostra, rate=16000)

In [ ]:
# Visualizando quantas amostras há em cada classe:
diretorio = 'C:/Users/Natanael/Documents/DidaticaTech/train/audio/'
classes=os.listdir(diretorio)
numero_audios=[]
for classe in classes: # para cada classe bird, cat, dog
    lista_nomes = [nome for nome in os.listdir(diretorio + classe) if nome.endswith('.wav')] # carrega todos os nomes dos arquivos de cada pasta para dentro da lista
    numero_audios.append(len(lista_nomes)) # salva o tamanho de cada pasta, ou seja, a quantidade de arquivos presentes nela
    
print('Classe bird:', numero_audios[0], '\nClasse cat:', numero_audios[1], '\nClasse dog:', numero_audios[2],)

In [ ]:
# Mostrando um histograma da duração de cada arquivo
classes=["bird", "cat", "dog"]
duracao_gravacoes=[]
for classe in classes:
    lista_nomes = [nome for nome in os.listdir(diretorio + classe) if nome.endswith('.wav')]
    for nome in lista_nomes: # para cada arquivo
        amostra, sample_rate = librosa.load(diretorio + '/' + classe + '/' + nome, sr = 16000)
        duracao_gravacoes.append(float(len(amostra)/sample_rate)) # calcula a duração de cada arquivo (resultado da conta 'número_de_amostras_do_arquivo'/'número_de_amostras/segundo')
    
plt.hist(np.array(duracao_gravacoes))

In [ ]:
# Carregando todos os arquivos por meio da biblioteca librosa e fixando o sample_rate em 8.000 (para reduzir a quantidade de dados)
# Excluiremos as amostras que tiverem duração inferior a 1 segundo
todos_audios = []
todos_rotulos = []
cont=0
for classe in classes:
    print('Processando',classe)
    lista_nomes = [nome for nome in os.listdir(diretorio + '/'+ classe) if nome.endswith('.wav')]
    for nome in lista_nomes:
        cont+=1
        amostra, sample_rate = librosa.load(diretorio + '/' + classe + '/' + nome, sr = 16000)
        amostra = librosa.resample(amostra, sample_rate, 8000)
        if(len(amostra)== 8000): 
            todos_audios.append(amostra)
            todos_rotulos.append(classe)

In [ ]:
# Mostrando a quantidade total de amostras que ficamos e quanto havia
print('Ficamos com', len(todos_audios), 'de um total de', cont)

In [ ]:
# Obs: com esses dados, poderíamos criar um classificador usando redes neurais densas, por exemplo. Mas como iremos usar LSTM, 
# 8.000 timesteps é um número muito grande, então trabalhar com MFCC será útil.

In [ ]:
# Criando os espectrogramas MFCC
mfccs = []
for audio in todos_audios:
    mfcc = librosa.feature.mfcc(y=audio, sr=8000, n_mfcc=20)
    mfcc = np.transpose(mfcc) # invertendo a posição das features e timesteps, pois a LSTM espera receber nessa ordem: (timesteps, features)
    mfccs.append(mfcc)

In [ ]:
# mfccs é uma lista de arrays de duas dimensões. Precisamos transformar isso em um array numpy de 3 dimensões:
mfccs

In [ ]:
x = np.stack(mfccs, axis=0)

In [ ]:
# Visualizando o resultado disso:
x

In [ ]:
x.shape

In [ ]:
# Agora iremos trabalhar com as classes. Já criamos uma classe para cada amostra, vamos conferir:
len(todos_rotulos)

In [ ]:
todos_rotulos

In [ ]:
# Fazendo label encoding com essas classes:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y=le.fit_transform(todos_rotulos)
classes= list(le.classes_)

In [ ]:
# Aplicando one hot encoding:
from keras.utils import np_utils
y=np_utils.to_categorical(y, num_classes=len(classes))

In [ ]:
y.shape

In [ ]:
# Agora que já temos nossos dados x e y como arrays numpy nas dimensões corretas, podemos separar os dados entre treino e teste:
from sklearn.model_selection import train_test_split
x_treino, x_teste, y_treino, y_teste = train_test_split(x, y, stratify=y, test_size = 0.2) # stratify mantém a proporção das classes

In [ ]:
x_treino.shape

In [ ]:
x_teste.shape

In [ ]:
# Criando a LSTM
from keras.models import Sequential
from keras.layers import Dense, Embedding, LSTM, Dropout
import warnings
warnings.filterwarnings('ignore')

# Criando o modelo LSTM
modelo = Sequential()
modelo.add(LSTM(150, dropout=0.3, input_shape = (x_treino.shape[1], x_treino.shape[2])))
modelo.add(Dense(3, activation='softmax'))
modelo.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
modelo.fit(x_treino, y_treino, epochs = 200, batch_size = 500, validation_data=(x_teste, y_teste), verbose = 2)

In [ ]:
# Fazendo previsões:
from numpy import expand_dims
amostra = expand_dims(x_teste[0], 0) # criando uma dimensão extra para ficar (n_amostras, timesteps, features), pois ao pegar somente x_teste[0] a primeira dimensão morre
prob=modelo.predict(amostra) # obtendo as probabilidades de cada classe
index=np.argmax(prob[0]) # obtendo o índice da coluna com maior probabilidade
classes[index] # mostrando qual a respectiva classe

In [ ]:
# Verificando a resposta
index=np.argmax(y_teste[0])
classes[index]